In [ ]:
import os
import pandas as pd
import random
from PIL import Image
import torch
import numpy as np
from torch.utils.data import Dataset, DataLoader
import torchvision
from data_augmentation import get_train_transform, get_test_transform
from data_attack import get_attack_transform
from data_loader import get_dataloader, get_train_test_loaders, get_unlabeled_loader
import matplotlib.pyplot as plt
from torchvision import transforms
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from FGSM_attack import fgsm_attack, denorm, test

In [ ]:
torch.cuda.is_available()

In [ ]:
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')

# Assuming that we are on a CUDA machine, this should print a CUDA device:

print(device)

In [ ]:
csv_path = '/home/hice1/jguardia7/scratch/datasets/alessandrasala79/ai-vs-human-generated-dataset/versions/4/train.csv'
image_folder = '/home/hice1/jguardia7/scratch/datasets/alessandrasala79/ai-vs-human-generated-dataset/versions/4/train_data'

In [ ]:
train_loader, test_loader, attack_loader, train_dataset, test_dataset, attack_dataset = get_train_test_loaders(csv_path, image_folder, test_batch_size=1, augmentation=False, attack_style='SaltAndPepper')
#attack style options: SaltAndPepper, Posterize, GaussNoise, RandomShadow
for images, labels, _ in train_loader:
        print("Train batch shape:", images.shape)
        print("Train labels shape:", labels.shape)
        break

for images, labels, _ in test_loader:
    print("Test batch shape:", images.shape)
    print("Test labels shape:", labels.shape)
    break
    
for images, labels, _ in attack_loader:
    print("Attack test batch shape:", images.shape)
    print("Attack test labels shape:", labels.shape)
    break
    
print(f"Total images in train set: {len(train_dataset)}")
print(f"Total images in test set: {len(test_dataset)}")
print(f"Total images in attack test set: {len(attack_dataset)}")

In [ ]:
def imshow(img):
    img = img / 2 + 0.5     # unnormalize
    npimg = img.numpy()
    plt.imshow(np.transpose(npimg, (1, 2, 0)))
    plt.show()

In [ ]:
dataiter = iter(train_loader)
images, labels, _ = next(dataiter)

In [ ]:
# show images
imshow(torchvision.utils.make_grid(images))
# print labels
classes = ('generated','real')
print(' '.join(f'{classes[labels[j]]:5s}' for j in range(32)))

In [ ]:
dataattack = iter(attack_loader)
imagesattack = next(dataattack)

# show images
imshow(torchvision.utils.make_grid(imagesattack[0]))

In [ ]:
#reference functions from https://pytorch.org/tutorials/beginner/blitz/cifar10_tutorial.html
#hand code 3 convolutions with normalization and relu, doubling channel count each block
#one fully connected layer that outputs two classes
#based on https://arxiv.org/pdf/1912.11035, Resnet is sufficient to train an AI detector
#we hand code the initial block similar to the initial resnet block based on code from
#https://www.digitalocean.com/community/tutorials/writing-resnet-from-scratch-in-pytorch

class ClassifierModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.block1 = nn.Sequential(
            nn.Conv2d(3, 6, 3),
            nn.BatchNorm2d(6),
            nn.ReLU()
        )
        self.block2 = nn.Sequential(
            nn.Conv2d(6, 12, 4),
            nn.BatchNorm2d(12),
            nn.ReLU()
        )
        self.block3 = nn.Sequential(
            nn.Conv2d(12, 24, 5),
            nn.BatchNorm2d(24),
            nn.ReLU()
        )
        self.pool = nn.MaxPool2d(2, 2)
        self.fc1 = nn.Linear(24 * 25 * 25, 2)


    def forward(self, x):
        x = self.pool(self.block1(x))
        x = self.pool(self.block2(x))
        x = self.pool(self.block3(x))
        x = torch.flatten(x, 1) # flatten all dimensions except batch
        x = self.fc1(x)
        return x


net = ClassifierModel().to(device)


In [ ]:
#cross entropy loss since we are training a classifier
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(net.parameters(), lr=0.001, momentum=0.9)

In [ ]:
#skip if model exists
if os.path.isfile('JuanchitoCNN.pth') != True:
    print('Model does not exist. Training...')
#if model does not exist, train model
    for epoch in range(4):  # loop over the dataset five times

        running_loss = 0.0
        for i, data in enumerate(train_loader, 0):
        # get the inputs; data is a list of [inputs, labels]
            inputs, labels = data[0].to(device), data[1].to(device)
        

        # zero the parameter gradients
            optimizer.zero_grad()

        # forward + backward + optimize
            outputs = net(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
        # print statistics
            running_loss += loss.item()
            if i % 200 == 199:    # print every 200 mini-batches
                print(f'[{epoch + 1}, {i + 1:5d}] loss: {running_loss / 2000:.3f}')
                running_loss = 0.0

    print('Finished Training. Saving...')
    #if model does not exist, save model state directory
    torch.save(net.state_dict(), 'JuanchitoCNN.pth')
    torch.save(net, 'JuanchitoCNN full.pth')
    print('Saved.')
else:
    print('Model exists. Loading weights')
    net.load_state_dict(torch.load('JuanchitoCNN.pth', weights_only=True))
    net.eval()
    net_full=torch.load('JuanchitoCNN full.pth', weights_only=False)
    net_full.eval()
    print('Weights loaded')

In [ ]:
correct = 0
total = 0
with torch.no_grad():
    for i, data in enumerate(test_loader, 0):
        images, labels = data[0].to(device), data[1].to(device)
        # calculate outputs by running images through the network
        outputs = net_full(images)
        # the class with the highest energy is what we choose as prediction
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
        if i % 200 == 199:    # print every 200 mini-batches
            print(f'[{i + 1:5d}]')

In [ ]:
print(f'Accuracy of the network on the test images: {100 * correct // total} %')

In [ ]:
#_, test_loader, _, _, test_dataset, _ = get_train_test_loaders(csv_path, image_folder, augmentation=False, attack_style='SaltAndPepper', batch_size=1)
accuracies = []
examples = []
epsilons = [0, .05, .1, .15, .2, .25, .3]
# Run test for each epsilon
for eps in epsilons:
    acc, ex = test(net_full, device, test_loader, eps)
    accuracies.append(acc)
    examples.append(ex)
    print(eps,'done')

In [ ]:
plt.figure(figsize=(5,5))
plt.plot(epsilons, accuracies, "*-")
plt.yticks(np.arange(0, 1.1, step=0.1))
plt.xticks(np.arange(0, .35, step=0.05))
plt.title("Accuracy vs Epsilon")
plt.xlabel("Epsilon")
plt.ylabel("Accuracy")
plt.show()

In [ ]:
# Plot several examples of adversarial samples at each epsilon
cnt = 0
plt.figure(figsize=(8,10))
for i in range(len(epsilons)):
    for j in range(len(examples[i])):
        cnt += 1
        plt.subplot(len(epsilons),len(examples[0]),cnt)
        plt.xticks([], [])
        plt.yticks([], [])
        if j == 0:
            plt.ylabel(f"Eps: {epsilons[i]}", fontsize=14)
        orig,adv,ex = examples[i][j]
        plt.title(f"{orig} -> {adv}")
        plt.imshow(np.moveaxis(ex, 0, -1))
plt.tight_layout()
plt.show()